# Inspired Final Model v1: Weak Correction 전략 극대화

이 노트북은 `submission_holiday_dinner_stronger_annotated`의 철학을 계승합니다.
**"메뉴와 날씨를 학습에 직접 넣으면 모델이 혼란스러워한다. 대신 보조 수단으로만 쓰자."**

**핵심 로직:**
1. **베이스 모델:** 메뉴/날씨를 제외한 운영 데이터로만 '참여율' 예측
2. **메뉴 보정:** 학습 데이터에서 메뉴별 오차(Residual)를 계산해 예측값에 소량 반영
3. **날씨 보정:** 비, 더위, 추위 정보를 '약한 시그널'로만 활용해 결과값 보정
4. **연휴 보정:** 연휴 전/후의 패턴을 상수로 보정

In [ ]:
import os
import re
import random
import warnings
import numpy as np
import pandas as pd
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
# 경로 자동 탐색 (CY2 환경)
def find_path(cands): 
    for p in cands: 
        if os.path.exists(p): return p
    return None

train_path = find_path([r"C:\Users\CY2\github\SecondProject\data\train_median.csv", r"./data/train_median.csv"])
test_path = find_path([r"C:\Users\CY2\github\SecondProject\data\test.csv", r"./data/test.csv"])
sub_path = find_path([r"C:\Users\CY2\github\SecondProject\data\sample_submission.csv", r"./data/sample_submission.csv"])
w_train_path = find_path([r"C:\Users\CY2\github\SecondProject\data\weather.csv", r"./data/weather.csv"])
w_test_path = find_path([r"C:\Users\CY2\github\SecondProject\data\weatherTest.csv", r"./data/weatherTest.csv"])

train = pd.read_csv(train_path, encoding="utf-8-sig")
test = pd.read_csv(test_path, encoding="utf-8-sig")
sample_submission = pd.read_csv(sub_path, encoding="utf-8-sig")
w_train = pd.read_csv(w_train_path, encoding="utf-8-sig")
w_test = pd.read_csv(w_test_path, encoding="utf-8-sig") if w_test_path else pd.DataFrame()

train["일자"] = pd.to_datetime(train["일자"])
test["일자"] = pd.to_datetime(test["일자"])

In [ ]:
# 날씨 데이터 통합 (보정용으로만 사용)
def prep_w(df):
    if df.empty: return df
    c = df.columns
    d_col, t_col, r_col = [x for x in c if "일시" in x or "날짜" in x][0], [x for x in c if "기온" in x][0], [x for x in c if "강수량" in x][0]
    res = df[[d_col, t_col, r_col]].copy()
    res.columns = ["일자", "기온", "강수량"]
    res["일자"] = pd.to_datetime(res["일자"])
    res["기온"] = pd.to_numeric(res["기온"], errors="coerce").interpolate().ffill().bfill()
    res["강수량"] = pd.to_numeric(res["강수량"], errors="coerce").fillna(0)
    res["is_rain"] = (res["강수량"] > 0).astype(int)
    res["is_hot"] = (res["기온"] >= 28).astype(int)
    res["is_cold"] = (res["기온"] <= 5).astype(int)
    return res

weather_all = pd.concat([prep_w(w_train), prep_w(w_test)]).drop_duplicates("일자")
train = train.merge(weather_all, on="일자", how="left")
test = test.merge(weather_all, on="일자", how="left")

In [ ]:
# 피처 생성 (학습용 핵심 피처만)
def add_base_features(df):
    df = df.copy().sort_values("일자").reset_index(drop=True)
    df["월"] = df["일자"].dt.month
    df["일"] = df["일자"].dt.day
    df["요일"] = df["일자"].dt.weekday
    df["식사가능자수"] = (df["본사정원수"] - df["본사휴가자수"] - df["본사출장자수"] - df["현본사소속재택근무자수"]).clip(lower=1)
    
    # 연휴 정보 (보정용)
    prev_gap = df["일자"].diff().dt.days.fillna(1)
    next_gap = df["일자"].shift(-1).sub(df["일자"]).dt.days.fillna(1)
    df["holiday_after"] = (prev_gap >= 2).astype(int)
    df["holiday_before"] = (next_gap >= 2).astype(int)
    
    # 운영 정보
    df["days_to_month_end"] = ((df["일자"] + pd.offsets.MonthEnd(0)) - df["일자"]).dt.days
    df["is_fri"] = (df["요일"] == 4).astype(int)
    df["is_wed"] = (df["요일"] == 2).astype(int)
    df["log_overtime"] = np.log1p(df["본사시간외근무명령서승인건수"])
    return df

train = add_base_features(train)
test = add_base_features(test)
train["중식참여율"] = train["중식계"] / train["식사가능자수"]
train["석식참여율"] = train["석식계"] / train["식사가능자수"]

In [ ]:
# 베이스 모델 학습 (운영 피처로만)
lunch_f = ["월","일","요일","식사가능자수","본사출장자수","본사시간외근무명령서승인건수","is_fri","days_to_month_end"]
dinner_f = ["월","일","요일","식사가능자수","본사출장자수","log_overtime","is_wed"]

xgb_lunch = XGBRegressor(n_estimators=1000, learning_rate=0.05, max_depth=5, random_state=42)
xgb_dinner = XGBRegressor(n_estimators=1000, learning_rate=0.05, max_depth=5, random_state=42)

xgb_lunch.fit(train[lunch_f], train["중식참여율"])
xgb_dinner.fit(train[dinner_f], train["석식참여율"])

# 학습 데이터에서의 잔차(오차) 계산
train["lunch_res"] = train["중식참여율"] - xgb_lunch.predict(train[lunch_f])
train["dinner_res"] = train["석식참여율"] - xgb_dinner.predict(train[dinner_f])

In [ ]:
# 메뉴 기반 자동 Weak Correction
def get_menu_adj(train_df, menu_col, res_col):
    keyword_adj = {}
    keywords = ["돈까스", "제육", "치킨", "카레", "비빔밥", "짜장", "탕수육", "볶음밥", "오리", "생선"]
    for kw in keywords:
        mask = train_df[menu_col].str.contains(kw, na=False)
        if mask.sum() > 5:
            # 오차 평균의 35%만 보정값으로 사용 (Weak Correction)
            keyword_adj[kw] = train_df.loc[mask, res_col].mean() * 0.35
    return keyword_adj

l_menu_adj = get_menu_adj(train, "중식메뉴", "lunch_res")
d_menu_adj = get_menu_adj(train[train["석식계"]>0], "석식메뉴", "dinner_res")

def apply_menu_adj(menu_text, adj_map):
    adj = 0
    for kw, val in adj_map.items():
        if kw in str(menu_text): adj += val
    return np.clip(adj, -0.04, 0.04) # 너무 크게 흔들리지 않게 제한

In [ ]:
# 최종 예측 및 보정
p_l_ratio = xgb_lunch.predict(test[lunch_f])
p_d_ratio = xgb_dinner.predict(test[dinner_f])

l_adj = test["중식메뉴"].apply(lambda x: apply_menu_adj(x, l_menu_adj)).values
d_adj = test["석식메뉴"].apply(lambda x: apply_menu_adj(x, d_menu_adj)).values

# 날씨/연휴 보정 (Reference 값 반영)
w_l_sig = 0.010 * test["is_rain"] - 0.006 * test["is_hot"] + 0.004 * test["is_cold"]
w_d_sig = 0.004 * test["is_rain"] + 0.003 * test["is_cold"]
h_l_sig = -0.004 * test["holiday_before"] + 0.003 * test["holiday_after"]
h_d_sig = -0.005 * test["holiday_before"] + 0.003 * test["holiday_after"]

final_l_ratio = p_l_ratio + l_adj + w_l_sig + h_l_sig
final_d_ratio = p_d_ratio + d_adj + w_d_sig + h_d_sig

pred_l = final_l_ratio * test["식사가능자수"]
pred_d = final_d_ratio * test["식사가능자수"]

# 제출 파일 생성 (타입 안전)
submission = sample_submission.copy()
submission[submission.columns[1]] = np.clip(pred_l.values, 0, None)
submission[submission.columns[2]] = np.clip(pred_d.values, 0, None)

submission.to_csv("inspired_final_v1.csv", index=False, encoding="utf-8-sig")
print("저장 완료: inspired_final_v1.csv")